# Sample 02: OpenAI SDK ఇంటిగ్రేషన్

ఈ నోట్‌బుక్ OpenAI Python SDKతో అధునాతన ఇంటిగ్రేషన్‌ను ప్రదర్శిస్తుంది, Microsoft Foundry Local మరియు Azure OpenAI రెండింటినీ స్ట్రీమింగ్ ప్రతిస్పందనలతో మరియు సరైన లోపాల నిర్వహణతో మద్దతు ఇస్తుంది.

## అవలోకనం

ఈ నమూనా చూపిస్తుంది:
- Foundry Local మరియు Azure OpenAI మధ్య సులభమైన మార్పిడి
- మెరుగైన వినియోగదారు అనుభవం కోసం స్ట్రీమింగ్ చాట్ పూర్తి చేయడం
- FoundryLocalManager SDK యొక్క సరైన ఉపయోగం
- బలమైన లోపాల నిర్వహణ మరియుFallback యంత్రాంగాలు
- ఉత్పత్తి-సిద్ధమైన కోడ్ నమూనాలు


## ముందస్తు అవసరాలు

- **Foundry Local**: ఇన్‌స్టాల్ చేసి నడుస్తోంది (స్థానిక అంచనా కోసం)
- **Python**: 3.8 లేదా తరువాతి సంస్కరణ OpenAI SDK తో
- **Azure OpenAI**: చెల్లుబాటు అయ్యే ఎండ్‌పాయింట్ మరియు API కీ (క్లౌడ్ అంచనా కోసం)

### డిపెండెన్సీలు ఇన్‌స్టాల్ చేయండి


In [ ]:
# Install required packages
!pip install openai foundry-local-sdk

## లైబ్రరీలను దిగుమతి చేసుకోండి మరియు సెటప్ చేయండి


In [ ]:
import os
import sys
from openai import OpenAI
import time
from typing import Tuple

try:
    from foundry_local import FoundryLocalManager
    FOUNDRY_SDK_AVAILABLE = True
    print("✅ Foundry Local SDK is available")
except ImportError:
    FOUNDRY_SDK_AVAILABLE = False
    print("⚠️ Foundry Local SDK not available, manual configuration will be used")

## కాన్ఫిగరేషన్ ఎంపికలు

సరైన ఎన్విరాన్‌మెంట్ వేరియబుల్స్ సెట్ చేయడం ద్వారా Azure OpenAI (క్లౌడ్) లేదా Foundry Local (డివైస్‌పై) మధ్య ఎంచుకోండి.


### Option 1: Azure OpenAI కాన్ఫిగరేషన్

మీ Azure OpenAI క్రెడెన్షియల్స్‌ను అన్‌కామెంట్ చేసి సెట్ చేయండి:


In [ ]:
# Azure OpenAI Configuration
# Uncomment and set your actual values

# os.environ["AZURE_OPENAI_ENDPOINT"] = "https://your-resource.openai.azure.com"
# os.environ["AZURE_OPENAI_API_KEY"] = "your-api-key-here"
# os.environ["AZURE_OPENAI_API_VERSION"] = "2024-08-01-preview"
# os.environ["MODEL"] = "your-deployment-name"  # e.g., "gpt-4"

print("Azure OpenAI configuration ready (if credentials are set)")

### Option 2: Foundry స్థానిక కాన్ఫిగరేషన్

స్థానిక ఇన్ఫరెన్స్ కోసం డిఫాల్ట్ సెట్టింగులు:


In [ ]:
# Foundry Local Configuration (default)
FOUNDRY_MODEL = "phi-4-mini"  # Change to your preferred model
FOUNDRY_BASE_URL = "http://localhost:51211"
FOUNDRY_API_KEY = ""  # Usually empty for local

print(f"Foundry Local configuration ready with model: {FOUNDRY_MODEL}")

## క్లయింట్ ఫ్యాక్టరీ ఫంక్షన్లు

మీ కాన్ఫిగరేషన్ ఆధారంగా సరైన OpenAI క్లయింట్‌ను ఈ ఫంక్షన్లు సృష్టిస్తాయి:


In [ ]:
def create_azure_client() -> Tuple[OpenAI, str]:
    """Create Azure OpenAI client."""
    azure_endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT")
    azure_api_key = os.environ.get("AZURE_OPENAI_API_KEY")
    azure_api_version = os.environ.get("AZURE_OPENAI_API_VERSION", "2024-08-01-preview")
    
    if not azure_endpoint or not azure_api_key:
        raise ValueError("Azure OpenAI endpoint and API key are required")
    
    model = os.environ.get("MODEL", "your-deployment-name")
    client = OpenAI(
        base_url=f"{azure_endpoint}/openai",
        api_key=azure_api_key,
        default_query={"api-version": azure_api_version},
    )
    
    print(f"🌐 Azure OpenAI client created with model: {model}")
    return client, model


def create_foundry_client() -> Tuple[OpenAI, str]:
    """Create Foundry Local client with SDK management."""
    alias = FOUNDRY_MODEL
    
    if FOUNDRY_SDK_AVAILABLE:
        try:
            # Use FoundryLocalManager for proper service management
            print(f"🔄 Initializing Foundry Local with model: {alias}...")
            manager = FoundryLocalManager(alias)
            model_info = manager.get_model_info(alias)
            
            # Configure OpenAI client to use local Foundry service
            client = OpenAI(
                base_url=manager.endpoint,
                api_key=manager.api_key  # API key is not required for local usage
            )
            
            print(f"✅ Foundry Local SDK initialized")
            print(f"   Endpoint: {manager.endpoint}")
            print(f"   Model: {model_info.id}")
            return client, model_info.id
        except Exception as e:
            print(f"⚠️ Could not use Foundry SDK ({e}), falling back to manual configuration")
    
    # Fallback to manual configuration
    client = OpenAI(
        base_url=f"{FOUNDRY_BASE_URL}/v1",
        api_key=FOUNDRY_API_KEY
    )
    
    print(f"🔧 Manual Foundry Local configuration")
    print(f"   Endpoint: {FOUNDRY_BASE_URL}/v1")
    print(f"   Model: {alias}")
    return client, alias

## క్లయింట్ ప్రారంభించండి

ఇది ఆటోమేటిక్‌గా Azure OpenAI లేదా Foundry Local ఉపయోగించాలా అని గుర్తిస్తుంది:


In [ ]:
def initialize_client() -> Tuple[OpenAI, str, str]:
    """Initialize the appropriate OpenAI client."""
    
    # Check for Azure OpenAI configuration
    azure_endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT")
    azure_api_key = os.environ.get("AZURE_OPENAI_API_KEY")
    
    if azure_endpoint and azure_api_key:
        print("🌐 Azure OpenAI configuration detected")
        try:
            client, model = create_azure_client()
            return client, model, "azure"
        except Exception as e:
            print(f"❌ Azure OpenAI initialization failed: {e}")
            print("🔄 Falling back to Foundry Local...")
    
    # Use Foundry Local
    print("🏠 Using Foundry Local configuration")
    try:
        client, model = create_foundry_client()
        return client, model, "foundry"
    except Exception as e:
        print(f"❌ Foundry Local initialization failed: {e}")
        raise

# Initialize the client
print("Initializing OpenAI client...")
print("=" * 50)
client, model, provider = initialize_client()
print("=" * 50)
print(f"✅ Initialization complete! Using {provider} with model: {model}")

## ప్రాథమిక చాట్ పూర్తి

సాధారణ చాట్ పూర్తి పరీక్షించండి:


In [ ]:
def simple_chat(prompt: str, max_tokens: int = 150) -> str:
    """Send a simple chat message and get response."""
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {e}"

# Test basic chat
test_prompt = "Say hello from the SDK quickstart and explain what you are in one sentence."

print(f"👤 User: {test_prompt}")
print("\n🤖 Assistant:")
response = simple_chat(test_prompt)
print(response)

## స్ట్రీమింగ్ చాట్ పూర్తి

మంచి వినియోగదారు అనుభవం కోసం స్ట్రీమింగ్ ప్రతిస్పందనలను ప్రదర్శించండి:


In [ ]:
def streaming_chat(prompt: str, max_tokens: int = 300) -> str:
    """Send a chat message with streaming response."""
    try:
        print("🤖 Assistant (streaming):")
        
        stream = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            stream=True
        )
        
        full_response = ""
        for chunk in stream:
            if chunk.choices[0].delta.content is not None:
                content = chunk.choices[0].delta.content
                print(content, end="", flush=True)
                full_response += content
        
        print("\n")  # New line after streaming
        return full_response
    except Exception as e:
        error_msg = f"Error: {e}"
        print(error_msg)
        return error_msg

# Test streaming chat
streaming_prompt = "Explain the key benefits of using Microsoft Foundry Local for AI development. Include aspects like privacy, performance, and cost."

print(f"👤 User: {streaming_prompt}\n")
streaming_response = streaming_chat(streaming_prompt)

## బహుళ-తిరుగుడు సంభాషణ

సంభాషణ సందర్భాన్ని నిలుపుకోవడం ప్రదర్శించండి:


In [ ]:
class ConversationManager:
    """Manages multi-turn conversations with context."""
    
    def __init__(self, system_prompt: str = None):
        self.messages = []
        if system_prompt:
            self.messages.append({"role": "system", "content": system_prompt})
    
    def send_message(self, user_message: str, max_tokens: int = 200) -> str:
        """Send a message and get response while maintaining context."""
        # Add user message to conversation
        self.messages.append({"role": "user", "content": user_message})
        
        try:
            response = client.chat.completions.create(
                model=model,
                messages=self.messages,
                max_tokens=max_tokens
            )
            
            assistant_message = response.choices[0].message.content
            
            # Add assistant response to conversation
            self.messages.append({"role": "assistant", "content": assistant_message})
            
            return assistant_message
        except Exception as e:
            return f"Error: {e}"
    
    def get_conversation_length(self) -> int:
        """Get the number of messages in the conversation."""
        return len(self.messages)

# Create conversation manager with system prompt
system_prompt = "You are a helpful AI assistant specialized in explaining AI and machine learning concepts. Be concise but informative."
conversation = ConversationManager(system_prompt)

# Multi-turn conversation example
conversation_turns = [
    "What is the difference between AI inference on-device vs in the cloud?",
    "Which approach is better for privacy?",
    "What about performance and latency considerations?"
]

for i, turn in enumerate(conversation_turns, 1):
    print(f"\n{'='*60}")
    print(f"Turn {i}")
    print(f"{'='*60}")
    print(f"👤 User: {turn}")
    
    response = conversation.send_message(turn)
    print(f"\n🤖 Assistant: {response}")

print(f"\n📊 Conversation summary: {conversation.get_conversation_length()} messages total")

## పనితీరు తులన

విభిన్న పరిస్థితుల కోసం స్పందన సమయాలను తులన చేయండి:


In [ ]:
def benchmark_response_time(prompt: str, iterations: int = 3) -> dict:
    """Benchmark response time for a given prompt."""
    times = []
    responses = []
    
    for i in range(iterations):
        start_time = time.time()
        
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=50  # Keep responses short for timing
            )
            
            end_time = time.time()
            response_time = end_time - start_time
            
            times.append(response_time)
            responses.append(response.choices[0].message.content)
            
        except Exception as e:
            print(f"Error in iteration {i+1}: {e}")
    
    if times:
        avg_time = sum(times) / len(times)
        min_time = min(times)
        max_time = max(times)
        
        return {
            "average_time": avg_time,
            "min_time": min_time,
            "max_time": max_time,
            "all_times": times,
            "sample_response": responses[0] if responses else None
        }
    
    return {"error": "No successful responses"}

# Benchmark different types of prompts
benchmark_prompts = [
    "What is AI?",
    "Explain machine learning in simple terms.",
    "List 3 benefits of edge computing."
]

print(f"⏱️  Performance Benchmark ({provider} - {model})")
print("=" * 60)

for prompt in benchmark_prompts:
    print(f"\n📝 Prompt: '{prompt}'")
    results = benchmark_response_time(prompt)
    
    if "error" not in results:
        print(f"   ⏰ Average time: {results['average_time']:.2f}s")
        print(f"   ⚡ Fastest: {results['min_time']:.2f}s")
        print(f"   🐌 Slowest: {results['max_time']:.2f}s")
        print(f"   📄 Sample response: {results['sample_response'][:100]}...")
    else:
        print(f"   ❌ {results['error']}")

## అధునాతన కాన్ఫిగరేషన్ మరియు లోపాల నిర్వహణ

వివిధ పారామితులు మరియు లోపాల పరిస్థితులను పరీక్షించండి:


In [ ]:
def test_different_parameters():
    """Test chat completions with different parameters."""
    prompt = "Write a creative short story about AI."
    
    # Test different temperature values
    temperatures = [0.1, 0.5, 0.9]
    
    for temp in temperatures:
        print(f"\n🌡️ Temperature: {temp}")
        print("-" * 30)
        
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=100,
                temperature=temp
            )
            
            print(f"Response: {response.choices[0].message.content[:150]}...")
            
        except Exception as e:
            print(f"Error with temperature {temp}: {e}")

test_different_parameters()

## సర్వీస్ ఆరోగ్య తనిఖీ

సమగ్ర సర్వీస్ ఆరోగ్య మరియు సామర్థ్య తనిఖీ:


In [ ]:
def comprehensive_health_check():
    """Perform comprehensive health check of the service."""
    print("🏥 Comprehensive Health Check")
    print("=" * 50)
    
    # 1. Check model listing
    try:
        models_response = client.models.list()
        available_models = [m.id for m in models_response.data]
        print(f"✅ Model listing: SUCCESS")
        print(f"   📋 Available models: {available_models}")
        
        if model in available_models:
            print(f"   ✅ Current model '{model}' is available")
        else:
            print(f"   ⚠️ Current model '{model}' not found in available models")
    except Exception as e:
        print(f"❌ Model listing: FAILED - {e}")
    
    # 2. Test basic completion
    try:
        test_response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": "Say 'Health check successful'"}],
            max_tokens=10
        )
        print(f"✅ Basic completion: SUCCESS")
        print(f"   💬 Response: {test_response.choices[0].message.content}")
    except Exception as e:
        print(f"❌ Basic completion: FAILED - {e}")
    
    # 3. Test streaming
    try:
        stream = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": "Count to 3"}],
            max_tokens=20,
            stream=True
        )
        
        stream_content = ""
        chunk_count = 0
        for chunk in stream:
            if chunk.choices[0].delta.content:
                stream_content += chunk.choices[0].delta.content
                chunk_count += 1
        
        print(f"✅ Streaming: SUCCESS")
        print(f"   📦 Chunks received: {chunk_count}")
        print(f"   💬 Streamed content: {stream_content.strip()}")
    except Exception as e:
        print(f"❌ Streaming: FAILED - {e}")
    
    # 4. Provider-specific information
    print(f"\n📊 Configuration Summary:")
    print(f"   🏢 Provider: {provider}")
    print(f"   🤖 Model: {model}")
    if provider == "foundry":
        print(f"   🏠 Foundry SDK Available: {FOUNDRY_SDK_AVAILABLE}")
        print(f"   🔗 Base URL: {FOUNDRY_BASE_URL}")
    elif provider == "azure":
        print(f"   🌐 Azure Endpoint: {os.environ.get('AZURE_OPENAI_ENDPOINT', 'Not set')}")
        print(f"   🔑 API Version: {os.environ.get('AZURE_OPENAI_API_VERSION', 'Not set')}")

comprehensive_health_check()

## ఇంటరాక్టివ్ టెస్టింగ్

మీ స్వంత ప్రాంప్ట్‌లను ఇంటరాక్టివ్‌గా పరీక్షించడానికి ఈ సెల్‌ను ఉపయోగించండి:


In [ ]:
# Interactive testing - modify the prompt below
custom_prompt = "Explain the concept of 'edge AI' and why it's becoming important."
use_streaming = True  # Set to False for regular completion

print(f"👤 Custom Prompt: {custom_prompt}\n")

if use_streaming:
    custom_response = streaming_chat(custom_prompt, max_tokens=250)
else:
    custom_response = simple_chat(custom_prompt, max_tokens=250)
    print(f"🤖 Assistant: {custom_response}")

## సారాంశం మరియు తదుపరి దశలు

ఈ నోట్‌బుక్ లో ఆధునిక OpenAI SDK ఇంటిగ్రేషన్ ను ప్రదర్శించింది:

### ✅ కీ ఫీచర్లు కవర్ చేయబడ్డాయి

1. **బహుళ-ప్రొవైడర్ మద్దతు**: Azure OpenAI మరియు Foundry Local మధ్య సులభంగా మార్పిడి
2. **స్ట్రీమింగ్ ప్రతిస్పందనలు**: మెరుగైన UX కోసం రియల్-టైమ్ టోకెన్ ఉత్పత్తి
3. **సంభాషణ నిర్వహణ**: సందర్భంతో బహుళ-తిరుగుబాటు సంభాషణలు
4. **పనితీరు బెంచ్‌మార్కింగ్**: ప్రతిస్పందన సమయం కొలత మరియు విశ్లేషణ
5. **సమగ్ర ఆరోగ్య తనిఖీలు**: సేవా ధృవీకరణ మరియు నిర్ధారణలు
6. **లోపాల నిర్వహణ**: బలమైన లోపాల నిర్వహణ మరియు ఫాల్బ్యాక్ యంత్రాంగాలు

### 🏆 Foundry Local vs Azure OpenAI

| అంశం | Foundry Local | Azure OpenAI |
|--------|---------------|---------------|
| **గోప్యత** | ✅ డేటా లోకల్ లోనే ఉంటుంది | ⚠️ డేటా క్లౌడ్ కు పంపబడుతుంది |
| **విలంబం** | ✅ తక్కువ (లోకల్ ఇన్ఫరెన్స్) | ⚠️ ఎక్కువ (నెట్‌వర్క్ ఆధారితం) |
| **ఖర్చు** | ✅ ఉచితం (హార్డ్వేర్ తర్వాత) | 💰 టోకెన్ ప్రకారం చెల్లింపు |
| **ఆఫ్‌లైన్** | ✅ ఆఫ్‌లైన్ లో పనిచేస్తుంది | ❌ ఇంటర్నెట్ అవసరం |
| **మోడల్ వైవిధ్యం** | ⚠️ పరిమిత ఎంపిక | ✅ పూర్తి మోడల్ యాక్సెస్ |
| **స్కేలింగ్** | ⚠️ హార్డ్వేర్ ఆధారితం | ✅ అపరిమిత స్కేలింగ్ |

### 🚀 తదుపరి దశలు

- **సాంపిల్ 04**: Chainlit చాట్ అప్లికేషన్ నిర్మాణం
- **సాంపిల్ 05**: బహుళ-ఏజెంట్ ఆర్కెస్ట్రేషన్ సిస్టమ్స్
- **సాంపిల్ 06**: తెలివైన మోడల్ రౌటింగ్
- **ప్రొడక్షన్ డిప్లాయ్‌మెంట్**: స్కేలింగ్ మరియు మానిటరింగ్ పరిగణనలు

### 💡 ఉత్తమ ఆచారాలు

1. **ప్రొవైడర్ల మధ్య ఎప్పుడూ ఫాల్బ్యాక్ యంత్రాంగాలు అమలు చేయండి**
2. **దీర్ఘ ప్రతిస్పందనల కోసం స్ట్రీమింగ్ ఉపయోగించండి** పనితీరు మెరుగుదల కోసం
3. **ప్రొడక్షన్ అప్లికేషన్ల కోసం సరైన లోపాల నిర్వహణ అమలు చేయండి**
4. **వివిధ ప్రొవైడర్ల ప్రతిస్పందన సమయాలు మరియు ఖర్చులను పర్యవేక్షించండి**
5. **మీ నిర్దిష్ట అవసరాల ఆధారంగా సరైన ప్రొవైడర్ ఎంచుకోండి**


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్పష్టత**:  
ఈ పత్రాన్ని AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నించినప్పటికీ, ఆటోమేటెడ్ అనువాదాల్లో పొరపాట్లు లేదా తప్పిదాలు ఉండవచ్చు. మూల పత్రం దాని స్వదేశీ భాషలోనే అధికారిక మూలంగా పరిగణించాలి. ముఖ్యమైన సమాచారానికి, ప్రొఫెషనల్ మానవ అనువాదం సిఫార్సు చేయబడుతుంది. ఈ అనువాదం వాడకంలో ఏర్పడిన ఏవైనా అపార్థాలు లేదా తప్పుదారుల కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
